In [6]:
import os
import numpy as np
import glob
import re
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [ ]:

# ==========================================================
# PATHS
# ==========================================================
INPUT_DIR    = r"./../../dataFolders/MuscaChasingBeads/Body_Orientation"
SMOOTH_DIR   = r"./../../dataFolders/MuscaChasingBeads/xyz_smooth/"
FIG_BASE_DIR = r"./../../dataFolders/MuscaChasingBeads/Figures/Body_Orientation/"
os.makedirs(FIG_BASE_DIR, exist_ok=True)


In [13]:

# ==========================================================
# PATHS
# ==========================================================
INPUT_DIR    = r"./../../dataFolders/MuscaChasingBeads/Body_Orientation"
SMOOTH_DIR   = r"./../../dataFolders/MuscaChasingBeads/xyz_smooth/"
FIG_BASE_DIR = r"./../../dataFolders/MuscaChasingBeads/Figures/Body_Orientation/"
os.makedirs(FIG_BASE_DIR, exist_ok=True)

# Font size settings
TITLE_SIZE = 22
LABEL_SIZE = 18
TICK_SIZE  = 14

csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_ORIENTATION.csv")))

# ==========================================================
# PLOTTING
# ==========================================================
for path in csv_files:
    fname = os.path.basename(path)

    match      = re.search(r"(Trial\d+_\d+rpm)", fname)
    trial_name = match.group(1) if match else "Unknown_Trial"

    trial_save_dir = os.path.join(FIG_BASE_DIR, trial_name)
    os.makedirs(trial_save_dir, exist_ok=True)
    print(f"Generating Orientation plots for: {trial_name}")

    df     = pd.read_csv(path)
    frames = df["frame"].values

    def save_orientation_plot(columns, title, suffix, colors):
        plt.figure(figsize=(15, 8))
        for col, color in zip(columns, colors):
            label = col.split('_')[0].capitalize()
            plt.plot(frames, df[col], label=label, color=color, linewidth=2)
        plt.title(f"{title}: {trial_name}", fontsize=TITLE_SIZE, fontweight='bold', pad=20)
        plt.xlabel("Frame", fontsize=LABEL_SIZE)
        plt.ylabel("Angle (degrees)", fontsize=LABEL_SIZE)
        plt.xticks(fontsize=TICK_SIZE)
        plt.yticks(fontsize=TICK_SIZE)
        plt.legend(fontsize=TICK_SIZE, loc='upper right')
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.savefig(os.path.join(trial_save_dir, f"{trial_name}_{suffix}.png"),
                    dpi=300, bbox_inches='tight')
        plt.close()

    # 1. Body Orientation (Center-based)
    save_orientation_plot(
        ["yaw_center", "pitch_center", "roll_center"],
        "Body Orientation (Center-based)",
        "Orientation_Body",
        ["crimson", "dodgerblue", "forestgreen"]
    )
    # 2. Gaze/Target Orientation (Head-based)
    save_orientation_plot(
        ["yaw_head", "pitch_head", "roll_head"],
        "Target Orientation (Head-Based)",
        "Orientation_Gaze",
        ["darkorange", "purple", "black"]
    )

    # ================= 3D TRAJECTORY COLOURED BY ORIENTATION =================
    smooth_path = glob.glob(os.path.join(SMOOTH_DIR, f"{trial_name}*_SMOOTH.csv"))
    if not smooth_path:
        print(f"  WARNING: No SMOOTH file found for {trial_name} — skipping 3D plots")
        continue

    df_s = pd.read_csv(smooth_path[0])
    hx   = df_s["pt2_X"].values
    hy   = df_s["pt2_Y"].values
    hz   = df_s["pt2_Z"].values

    # 3 plots — one per angle
    angle_configs = [
        ("yaw_center",   "Yaw",   "RdBu",    "3D_yaw"),
        ("pitch_center", "Pitch", "RdYlGn",  "3D_pitch"),
        ("roll_center",  "Roll",  "Plasma",  "3D_roll"),
    ]

    for col, label, colorscale, suffix in angle_configs:
        angle_vals = df[col].values

        fig_3d = go.Figure(data=go.Scatter3d(
            x=hx, y=hy, z=hz,
            mode="lines+markers",
            name=f"Fly — {label}",
            marker=dict(
                size=3,
                color=angle_vals,
                colorscale=colorscale,
                colorbar=dict(title=f"{label} (°)"),
                showscale=True,
            ),
            line=dict(
                color=angle_vals,
                colorscale=colorscale,
                width=3,
            ),
            hovertemplate=(
                f"Frame: %{{text}}<br>"
                f"X: %{{x:.4f}}<br>"
                f"Y: %{{y:.4f}}<br>"
                f"Z: %{{z:.4f}}<br>"
                f"{label}: %{{marker.color:.2f}}°"
                "<extra></extra>"
            ),
            text=df_s["frame"].values,
        ))

        fig_3d.update_layout(
            title=f"{trial_name} — Trajectory coloured by {label} (Body)",
            scene=dict(
                xaxis_title="X",
                yaxis_title="Y",
                zaxis_title="Z",
            ),
            width=1100,
            height=750,
        )

        out_html = os.path.join(trial_save_dir, f"{trial_name}_{suffix}.html")
        fig_3d.write_html(out_html)
        print(f"  Saved: {out_html}")

    print(f"  All plots saved to: {trial_save_dir}")

print(f"\nAll Orientation plots saved in: {FIG_BASE_DIR}")

Generating Orientation plots for: Trial2_180rpm
  Saved: ./../../dataFolders/MuscaChasingBeads/Figures/Body_Orientation/Trial2_180rpm\Trial2_180rpm_3D_yaw.html
  Saved: ./../../dataFolders/MuscaChasingBeads/Figures/Body_Orientation/Trial2_180rpm\Trial2_180rpm_3D_pitch.html
  Saved: ./../../dataFolders/MuscaChasingBeads/Figures/Body_Orientation/Trial2_180rpm\Trial2_180rpm_3D_roll.html
  All plots saved to: ./../../dataFolders/MuscaChasingBeads/Figures/Body_Orientation/Trial2_180rpm
Generating Orientation plots for: Trial2_200rpm
  Saved: ./../../dataFolders/MuscaChasingBeads/Figures/Body_Orientation/Trial2_200rpm\Trial2_200rpm_3D_yaw.html
  Saved: ./../../dataFolders/MuscaChasingBeads/Figures/Body_Orientation/Trial2_200rpm\Trial2_200rpm_3D_pitch.html
  Saved: ./../../dataFolders/MuscaChasingBeads/Figures/Body_Orientation/Trial2_200rpm\Trial2_200rpm_3D_roll.html
  All plots saved to: ./../../dataFolders/MuscaChasingBeads/Figures/Body_Orientation/Trial2_200rpm
Generating Orientation plots